In [1]:
import air_quality

In [2]:
from pathlib import Path

In [3]:
import csv

In [4]:
BASE_DIR = Path.cwd()
DATA_FILE = BASE_DIR / "data" / "openaq_varna_8843.csv"

with DATA_FILE.open(newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    print(reader.fieldnames)

['location_id', 'location_name', 'parameter', 'value', 'unit', 'datetimeUtc', 'datetimeLocal', 'timezone', 'latitude', 'longitude', 'country_iso', 'isMobile', 'isMonitor', 'owner_name', 'provider']


In [5]:
print(*reader.fieldnames, sep="\n")

location_id
location_name
parameter
value
unit
datetimeUtc
datetimeLocal
timezone
latitude
longitude
country_iso
isMobile
isMonitor
owner_name
provider


In [6]:
print(dir(reader.fieldnames))

['__add__', '__class__', '__class_getitem__', '__contains__', '__delattr__', '__delitem__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__iadd__', '__imul__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__mul__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__reversed__', '__rmul__', '__setattr__', '__setitem__', '__sizeof__', '__str__', '__subclasshook__', 'append', 'clear', 'copy', 'count', 'extend', 'index', 'insert', 'pop', 'remove', 'reverse', 'sort']


In [7]:
sep="\n"

In [8]:
name = "Varna"

In [9]:
print(name)

Varna


In [10]:
datetime_text = "2026-06-01T00:00:00+03:00"

In [11]:
date_part, time_part = datetime_text.split("T")

In [12]:
print(datetime_text)

2026-06-01T00:00:00+03:00


In [13]:
import air_quality 
air_quality.format_datetime(datetime_text)

'01.06.2026 00:00'

In [14]:
year, month, day = date_part.split("-")

In [15]:
print (date_part)

2026-06-01


In [16]:
date_part = "2026-06-01"
year, month, day = date_part.split("-")
print(year)
print(month)
print(day)

2026
06
01


In [17]:
formatted_date = f"{day}.{month}.{year}"

In [18]:
print(formatted_date)

01.06.2026


In [19]:
formatted_time = time_part[:5]

In [20]:
print(formatted_time)

00:00


In [21]:
empty_result = air_quality.format_datetime("")

In [22]:
print(empty_result == "")

True


In [23]:
parameter_counts = {}

with DATA_FILE.open(newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    for row in reader:
        parameter = row.get("parameter", "").strip()

        if parameter:
            parameter_counts[parameter] = parameter_counts.get(parameter, 0) + 1

print(parameter_counts)

{'co': 600, 'no2': 600, 'o3': 600, 'pm10': 600, 'pm25': 600, 'so2': 600}


In [24]:
value_text = row.get("value", "")

print(value_text)
print(type(value_text))

11.38
<class 'str'>


In [25]:
print("Parameter:", row.get("parameter", ""))
print("Value:", row.get("value", ""))
print("Unit:", row.get("unit", ""))
print("UTC:", row.get("datetimeUtc", ""))
print("Local:", row.get("datetimeLocal", ""))

Parameter: so2
Value: 11.38
Unit: µg/m³
UTC: 2026-07-02T00:00:00Z
Local: 2026-07-02T03:00:00+03:00


In [26]:
latest_rows = {}

with DATA_FILE.open(newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    for row in reader:
        parameter = row.get("parameter", "").strip()
        datetime_utc = row.get("datetimeUtc", "").strip()

        if not parameter or not datetime_utc:
            continue

        if parameter not in latest_rows:
            latest_rows[parameter] = row

        elif datetime_utc > latest_rows[parameter]["datetimeUtc"]:
            latest_rows[parameter] = row

In [27]:
print(sorted(latest_rows))

['co', 'no2', 'o3', 'pm10', 'pm25', 'so2']


In [28]:
for parameter in sorted(latest_rows):
    latest_row = latest_rows[parameter]

    print(
        f"{parameter}: "
        f"{latest_row['value']} {latest_row['unit']} | "
        f"UTC: {latest_row['datetimeUtc']} | "
        f"Local: {latest_row['datetimeLocal']}"
    )

co: 280 µg/m³ | UTC: 2026-07-02T00:00:00Z | Local: 2026-07-02T03:00:00+03:00
no2: 41 µg/m³ | UTC: 2026-07-02T00:00:00Z | Local: 2026-07-02T03:00:00+03:00
o3: 62.91 µg/m³ | UTC: 2026-07-02T00:00:00Z | Local: 2026-07-02T03:00:00+03:00
pm10: 24.41 µg/m³ | UTC: 2026-07-02T00:00:00Z | Local: 2026-07-02T03:00:00+03:00
pm25: 10.61 µg/m³ | UTC: 2026-07-02T00:00:00Z | Local: 2026-07-02T03:00:00+03:00
so2: 11.38 µg/m³ | UTC: 2026-07-02T00:00:00Z | Local: 2026-07-02T03:00:00+03:00


In [29]:
blank_value_count = 0
non_numeric_value_count = 0
negative_value_count = 0
negative_examples = []

with DATA_FILE.open(newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    for row in reader:
        value_text = row.get("value", "").strip()

        if not value_text:
            blank_value_count += 1
            continue

        try:
            value = float(value_text)
        except ValueError:
            non_numeric_value_count += 1
            continue

        if value < 0:
            negative_value_count += 1

            if len(negative_examples) < 5:
                negative_examples.append(
                    {
                        "parameter": row.get("parameter", ""),
                        "value": value,
                        "datetimeUtc": row.get("datetimeUtc", ""),
                    }
                )

print("Blank values:", blank_value_count)
print("Non-numeric values:", non_numeric_value_count)
print("Negative values:", negative_value_count)
print("Negative examples:", negative_examples)

Blank values: 0
Non-numeric values: 0
Negative values: 142
Negative examples: [{'parameter': 'co', 'value': -1000.0, 'datetimeUtc': '2026-06-03T10:00:00Z'}, {'parameter': 'co', 'value': -1000.0, 'datetimeUtc': '2026-06-15T20:00:00Z'}, {'parameter': 'co', 'value': -1000.0, 'datetimeUtc': '2026-06-15T21:00:00Z'}, {'parameter': 'co', 'value': -1000.0, 'datetimeUtc': '2026-06-15T22:00:00Z'}, {'parameter': 'co', 'value': -1000.0, 'datetimeUtc': '2026-06-15T23:00:00Z'}]
